# Compresión de diccionarios e índices en Recuperación de Información

En este cuaderno vamos a explorar cómo comprimir tanto el **vocabulario de términos** como los **postings** de un índice invertido.


---





## 1️⃣ Compresión del Vocabulario

Existen varias formas de almacenar un diccionario:

1. **Fixed-width entries**  
   - Cada término ocupa un tamaño fijo (ej. 20 bytes) + 4 bytes frecuencia + 4 bytes puntero.  
   - Ventaja: acceso aleatorio rápido.  
   - Desventaja: desperdicio de espacio si los términos son cortos.

2. **Dictionary-as-a-string**  
   - Todos los términos concatenados en una cadena continua, y se almacenan punteros a su posición.  
   - Ventaja: reduce el espacio ocupado aproximadamente un 30%.

3. **Blocked storage**  
   - Agrupa k términos consecutivos en un bloque, guardando puntero solo al primer término y la longitud de cada término.  
   - Ventaja: reduce tamaño, aunque la búsqueda es un poco más lenta.

4. **Front coding**  
   - Muchos términos comparten prefijos.  
   - Ejemplo: "automat", "automatic", "automation" → guardamos el prefijo "automat" una sola vez y los sufijos ["", "ic", "ion"].


---



Se definen algunos documentos con términos relacionados con aprendizaje automático, algoritmos y recuperación de información.

In [ ]:
docs = [
    "El aprendizaje automático es útil",
    "El aprendizaje automáticamente mejora los resultados",
    "La automatización de procesos es fundamental en la industria moderna",
    "Debemos automatizar tareas repetitivas para ganar eficiencia",
    "El aprendizaje profundo es un tipo de aprendizaje automático",
    "Los algoritmos automáticos facilitan la búsqueda de información",
    "La recuperación de información se puede hacer con modelos probabilísticos",
    "BM25 es un modelo probabilístico avanzado que mejora BIM",
    "El aprendizaje automático se aplica automáticamente en sistemas inteligentes",
    "La automatización automática puede parecer redundante, pero tiene matices conceptuales",
    "Automatizar la recopilación de datos reduce errores humanos",
    "El aprendizaje supervisado y no supervisado son variantes del aprendizaje automático",
    "Los modelos automáticos de clasificación ayudan a la toma de decisiones",
    "Los sistemas automáticos requieren mantenimiento periódico",
    "La búsqueda automática de patrones es una tarea clásica en minería de datos",
    "La recuperación probabilística mejora la precisión de los resultados",
    "Los enfoques automáticos permiten procesar grandes volúmenes de texto",
    "Automáticamente se ajustan los parámetros del modelo según los datos",
    "El aprendizaje automático avanzado permite automatizar procesos de predicción",
    "La automatización inteligente integra aprendizaje automático y razonamiento simbólico"
]

El primer paso para generar el diccionario es extraer términos únicos que se usen a modo de vocabulario.

In [ ]:
# ============================
# Extraer términos únicos
# ============================

# Convertir todo a minúsculas y separar palabras
terms = set()
for doc in docs:
    for word in doc.lower().split():
        terms.add(word)
terms = sorted(list(terms))  # ordenar alfabéticamente

### 0. Vocabulario sin compresión

In [ ]:
# ==========================
# Diccionario original (sin compresión)
# ==========================

# Convertimos los términos a bytes concatenados, sin compresión
vocab_original_bytes = ''.join(terms).encode('utf-8')
size_original = len(vocab_original_bytes)

print("Diccionario original (sin comprimir):", size_original, "bytes")

### 1. Fixed-width entries

Cada término ocupa un tamaño fijo, por ejemplo 20 bytes.

Cada entrada añade 4 bytes para frecuencia y 4 bytes para puntero.

Ventaja: acceso aleatorio rápido.

Desventaja: desperdicia espacio si el término es corto.

In [ ]:
# ==========================
# Diccionario con fixed entries
# ==========================

fixed_width_size_term = 20 + 4 + 4  # 20 bytes término + 4 frecuencia + 4 puntero
size_fixed_width = fixed_width_size_term * len(terms)

print("\nFixed-width entries:")
# Simulación de almacenamiento: rellenar con espacios para cada término
stored_fixed_width = []
for term in terms:
    term_bytes = term.ljust(10)  # rellenar hasta 20 bytes
    stored_fixed_width.append((term_bytes, 1, 0))  # (término, frecuencia, puntero)
    print(f"{term} -> {fixed_width_size_term} bytes")
print("Tamaño total:", size_fixed_width, "bytes")


Vamos a implementar la decodificación para que puedas reconstruir los términos originales a partir del fixed entries.

In [ ]:
# Decodificación
stored_fixed_width = [(term.ljust(20), 1, 0) for term in terms]
decoded_fixed_width = [t[0].strip() for t in stored_fixed_width]
assert decoded_fixed_width == terms
print("Decodificación Fixed-width entries:")
print(decoded_fixed_width)
print("Decodificación correcta:", decoded_fixed_width == terms)

### 2. Dictionary-as-a-string

Todos los términos concatenados en una cadena continua.

Se almacenan punteros a la posición inicial de cada término.

Ventaja: reduce el espacio (~30%) en comparación con fixed-width.

In [ ]:
# ==========================
# Dictionary-as-a-string
# ==========================

# Concatenar todos los términos
dict_string = ''.join(terms)

# Punteros a la posición inicial de cada término
pointers = []
pos = 0
for term in terms:
    pointers.append(pos)
    pos += len(term)

size_dict_string = len(dict_string.encode('utf-8')) + len(pointers)*4  # punteros de 4 bytes
print("\nDictionary-as-a-string:")
print("Cadena concatenada:", dict_string)
print("Punteros:", pointers)
print("Tamaño total aproximado:", size_dict_string, "bytes")


Vamos a implementar la decodificación para que puedas reconstruir los términos originales a partir del Dictionary-as-string.

In [ ]:
#Decodificación
decoded_string_dict = [dict_string[pointers[i]:pointers[i]+len(terms[i])] for i in range(len(terms))]
assert decoded_string_dict == terms
print("Decodificación Dictionary-as-a-string:")
print(decoded_string_dict)
print("Decodificación correcta:", decoded_string_dict == terms)

### 3. Blocked storage

Se agrupan k términos consecutivos en un bloque.

Se guarda puntero solo al primer término del bloque y la longitud de cada término.

Ventaja: reduce el tamaño, aunque la búsqueda es ligeramente más lenta.

In [ ]:
# ==========================
# Diccionario con Blocked storage
# ==========================

k = 20  # tamaño del bloque
blocks = []

for i in range(0, len(terms), k):
    block_terms = terms[i:i+k]
    # puntero al primer término del bloque (su posición en dict_string)
    block_pointer = pointers[i]
    # longitudes de cada término
    lengths = [len(term) for term in block_terms]
    blocks.append((block_pointer, lengths, block_terms))

# Calcular tamaño aproximado: puntero + longitudes + suma de bytes de términos
size_blocked = 0
for ptr, lengths, block_terms in blocks:
    size_blocked += 4  # puntero
    size_blocked += 4 * len(lengths)  # longitudes
    size_blocked += sum(len(term.encode('utf-8')) for term in block_terms)

print("\nBlocked storage (k=3):")
for block in blocks:
    print(block)
print("Tamaño total aproximado:", size_blocked, "bytes")


Vamos a implementar la decodificación para que puedas reconstruir los términos originales a partir del Blocked storage.

In [ ]:
# Decodificación
decoded_blocked = []
for _, lengths, block_terms in blocks:
    decoded_blocked.extend(block_terms)
assert decoded_blocked == terms
print("Decodificación Blocked storage:")
print(decoded_blocked)
print("Decodificación correcta:", decoded_blocked == terms)

### 4. Front coding

Muchos términos comparten prefijos.  

Ejemplo: "automat", "automatic", "automation" → guardamos el prefijo "automat" una sola vez y los sufijos ["", "ic", "ion"].

In [ ]:
# ==========================
# Diccionario con Front coding
# ==========================

terms.sort()
front_coded = []

front_coded = []
# Primer término completo
front_coded.append((0, terms[0]))

for i in range(1, len(terms)):
    prev_term = terms[i-1]
    term = terms[i]
    j = 0
    while j < min(len(prev_term), len(term)) and prev_term[j] == term[j]:
        j += 1
    sufijo = term[j:]
    front_coded.append((j, sufijo))

# Tamaño aproximado: suma de prefijos (como 1 byte) + sufijos
size_front_coding = sum([1 + len(sufijo.encode('utf-8')) for _, sufijo in front_coded])

# Mostrar resultado
print("Términos originales ordenados:")
print(terms)
print("\nFront Coding (longitud prefijo, sufijo):")
for fc in front_coded:
    print(fc)
print("Tamaño aproximado:", size_front_coding, "bytes")


Cada término se representa como un par `(longitud_prefijo, sufijo)`.
Por ejemplo, si tenemos los términos:

```
automático
automáticamente
automatización
```
La codificación podría ser:

```
(0, "automático")          # primero se guarda completo
(8, "mente")               # comparte los 8 primeros caracteres "automático"
(7, "ización")             # comparte los 7 primeros caracteres "automat"
```
Esto permite guardar el prefijo solo una vez y reducir significativamente el tamaño del diccionario.


Vamos a implementar la decodificación para que puedas reconstruir los términos originales a partir del Front Coding

In [ ]:
# Decodificación
def decode_front_coding(fc):
    decoded = []
    for i, (pref_len, suf) in enumerate(fc):
        if i == 0:
            decoded.append(suf)
        else:
            prev = decoded[-1]
            decoded.append(prev[:pref_len] + suf)
    return decoded

decoded_fc = decode_front_coding(front_coded)
assert decoded_fc == terms

print("Términos reconstruidos desde Front Coding:")
print(decoded_fc)
print("Decodificación correcta:", decoded_fc == terms)


### 5. Comparación de tamaño del vocabulario

In [ ]:
# ==========================
# Comparación gráfica
# ==========================
import matplotlib.pyplot as plt

labels = ["Original", "Front Coding", "Fixed-width", "Dict-as-string", "Blocked"]
sizes = [size_original, size_front_coding, size_fixed_width, size_dict_string, size_blocked]

plt.figure(figsize=(10,6))
bars = plt.bar(labels, sizes, color=['red','green','blue','orange','purple'])
plt.ylabel("Tamaño en bytes")
plt.title("Comparación de tamaño de diccionario: original vs métodos de compresión")
plt.xticks(rotation=30, ha='right')

for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2.0, yval + 5, int(yval), ha='center', va='bottom')

plt.show()

# **Ejercicios**

1. Comprobar qué ocurre cuando se cambia la cantidad de bytes en el caso de fixed-width

2. Comprobar qué ocurre si cambiamos el tamaño de bloques en Blocked.


## 2️⃣ Compresión de los postings



### 0. Crear un índice invertido simple

Creamos un índice invertido donde cada término apunta a la lista de documentos donde aparece.

In [ ]:

def create_inverted_index(docs):
    inverted_index = {}
    for doc_id, doc in enumerate(docs):
        for term in set(doc.lower().split()):
            if term not in inverted_index:
                inverted_index[term] = []
            inverted_index[term].append(doc_id)
    return inverted_index

index = create_inverted_index(docs)
print("\nÍndice invertido original:")
for term, postings in index.items():
    print(f"{term}: {postings}")


### 1. Compresión de Postings con Gaps

Cada posting se guarda como la diferencia respecto al documento anterior.
Ventaja: reduce el tamaño si los documentos están ordenados.

In [ ]:
# Función para convertir postings a gaps
def gaps(postings):
    if not postings:
        return []
    g = [postings[0]]
    for i in range(1, len(postings)):
        g.append(postings[i] - postings[i-1])
    return g

index_gaps = {term: gaps(p) for term, p in index.items()}
print("\nÍndice con gaps:")
for term, g in index_gaps.items():
    print(f"{term}: {g}")

# Tamaño aproximado (suma de enteros en bytes, 4 bytes por número)
size_gaps = sum([len(g) * 4 for g in index_gaps.values()])


In [ ]:
# Decodificación
def decode_gaps(gaps_list):
    decoded = []
    for g in gaps_list:
        if not decoded:
            decoded.append(g)
        else:
            decoded.append(decoded[-1] + g)
    return decoded

# Verificar decodificación
for term, g in index_gaps.items():
    assert decode_gaps(g) == index[term]

### 2. Variable-Byte Encoding (VB)

Codificamos cada número usando un esquema de bytes variable.  
Cada byte indica si es el último de la codificación sumando 128 al último byte.

In [ ]:
# Función simple para codificar un número en bytes
def vb_encode_number(number):
    bytes_list = []
    while True:
        bytes_list.insert(0, number % 128)
        if number < 128:
            break
        number //= 128
    bytes_list[-1] += 128  # marcar fin
    return bytes_list

def vb_encode_list(numbers):
    return [b for n in numbers for b in vb_encode_number(n)]

index_vb = {term: vb_encode_list(g) for term, g in index_gaps.items()}
size_vb = sum([len(vb) for vb in index_vb.values()])  # tamaño en bytes
print("\nÍndice con Variable-Byte encoding:")
for term, vb in index_vb.items():
    print(f"{term}: {vb}")


In [ ]:
# Decodificación
def vb_decode_list(vb_bytes):
    numbers = []
    n = 0
    for b in vb_bytes:
        if b < 128:
            n = n * 128 + b
        else:
            n = n * 128 + (b - 128)
            numbers.append(n)
            n = 0
    return numbers

# Verify decoding
for term, vb in index_vb.items():
    decoded = vb_decode_list(vb)
    # Decode the gaps back to original postings to compare
    decoded_gaps = decode_gaps(decoded)
    assert decoded_gaps == index[term]

print("Variable-Byte decoding successful.")

### 3. Gamma Codes

Cada número se codifica en bits usando la codificación Gamma.  
Se calcula tamaño aproximado en bits.





In [ ]:
def gamma_encode_number(n):
    if n == 0:
        return '0'
    import math
    L = int(math.floor(math.log2(n)))
    offset = bin(n)[3:]  # quitar '0b1'
    length = '0' * L + '1'
    return length + offset

index_gamma = {term: [gamma_encode_number(g) for g in gaps] for term, gaps in index_gaps.items()}
# Tamaño aproximado en bits
size_gamma = sum([sum(len(gamma) for gamma in gammas) for gammas in index_gamma.values()])
print("\nÍndice con Gamma codes:")
for term, gamma in index_gamma.items():
    print(f"{term}: {gamma}")

In [ ]:
def gamma_decode_number(code):
    i = 0
    numbers = []
    while i < len(code):
        L = 0
        while code[i] == '0':
            L += 1
            i += 1
        i += 1  # saltar el 1
        offset = code[i:i+L]
        i += L
        if offset == '':
            n = 2**L
        else:
            n = 2**L + int(offset, 2)
        numbers.append(n)
    return numbers

### 4. Comparación de tamaño de los postings

Comparamos el tamaño aproximado de los postings:

- Original (4 bytes por entero)
- Gaps (delta encoding)
- Variable-Byte (VB)
- Gamma Codes (bits)

In [ ]:
labels = ["Original (4 bytes)", "Gaps", "Variable-Byte", "Gamma (bits)"]
sizes = [
    sum([len(postings)*4 for postings in index.values()]),
    size_gaps,
    size_vb,
    size_gamma
]

import matplotlib.pyplot as plt

plt.figure(figsize=(10,6))
bars = plt.bar(labels, sizes, color=['red','green','blue','purple'])
plt.ylabel("Tamaño aproximado")
plt.title("Comparación de compresión de postings")
plt.xticks(rotation=30, ha='right')

for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2.0, yval + 0.5, int(yval), ha='center', va='bottom')

plt.show()

# **Ejercicios:**

1. Implementar las combinaciones de vocabularios con postings y mostrar el tamaño total.

2. ¿Qué combinación logra el mayor ahorro de espacio?

3. ¿Qué método de postings es más eficiente cuando los documentos son cortos?

4. ¿Cuánto espacio ocupa el vocabulario frente a los postings?

5. ¿Qué método sacrifica más acceso rápido a cambio de compresión?